# Lab 1 · Take-home notebook — evaluate before you trust ⏱ ~2–3 h at home

Demos prove a model *can* work. **Evaluation** shows *how often* and *where* it fails — and it is the part of your final project
that graders (and real users) care about most.

**Tasks in this notebook**
* **A.** Evaluate the sentiment model from the lab on 48 labelled reviews, overall and **by difficulty tag**.
* **B.** Evaluate a *second*, multilingual model and compare fairly (the label spaces differ: 5 stars vs. 2 classes!).
* **C.** Read two model cards and fill in the critique table.

The rest of the assignment (extending TextLab, written report) is in `assignment/ASSIGNMENT.md`.

In [1]:
import os, warnings; warnings.filterwarnings("ignore")
import pandas as pd, torch
from transformers import pipeline
from sklearn.metrics import confusion_matrix, classification_report

DEVICE = 0 if torch.cuda.is_available() else -1
MODEL_A = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"      # English, 2 labels, ~66M params
MODEL_B = "nlptown/bert-base-multilingual-uncased-sentiment"                 # 6 languages, 1-5 stars, ~167M params

DATA = next(p for p in ["../data/mini_reviews.csv", "data/mini_reviews.csv", "mini_reviews.csv"] if os.path.exists(p))
df = pd.read_csv(DATA)
print(df.shape)
print(df.label.value_counts().to_dict(), "\n", df.tag.value_counts().to_dict())
df.sample(5, random_state=1)

W1022 21:50:17.627000 54532 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


(48, 3)
{'NEGATIVE': 26, 'POSITIVE': 22} 
 {'easy': 16, 'negation': 10, 'slang': 6, 'other-language': 6, 'mixed': 6, 'sarcasm': 4}


,text,label,tag
33,"Meh, this show is mid at best and honestly kin...",NEGATIVE,slang
23,The battery doesn't last and it isn't even fast.,NEGATIVE,negation
38,"Ein wunderbares Hotel, ich komme sehr gerne wi...",POSITIVE,other-language
36,"Un film magnifique, je l'ai adoré du début à l...",POSITIVE,other-language
2,"Best pizza I have had in years, we will defini...",POSITIVE,easy


## A · Evaluate model A
First, predictions for the whole table (one batched call — remember Notebook 1).

In [2]:
clf_a = pipeline("sentiment-analysis", model=MODEL_A, device=DEVICE)
res = clf_a(df.text.tolist(), batch_size=16, truncation=True)
df["pred_a"] = [r["label"] for r in res]
df["conf_a"] = [r["score"] for r in res]
df[["text", "label", "pred_a", "conf_a"]].head()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 14520.89it/s]

,text,label,pred_a,conf_a
0,"Absolutely loved this movie, the acting was br...",POSITIVE,POSITIVE,0.999884
1,The hotel was spotless and the staff were incr...,POSITIVE,POSITIVE,0.999787
2,"Best pizza I have had in years, we will defini...",POSITIVE,POSITIVE,0.999792
3,Great battery life and the screen is gorgeous.,POSITIVE,POSITIVE,0.999887
4,A heartwarming story that left the whole theat...,POSITIVE,POSITIVE,0.999854


### TODO A1 — accuracy overall and per tag
Return a `pd.Series` indexed by tag (plus an `"ALL"` entry for the overall accuracy), rounded to 3 decimals, sorted from worst to best tag.
*(Hint: `(df[pred_col] == df["label"])` is a boolean Series; `groupby(df["tag"]).mean()`.)*

In [3]:
def accuracy_by_tag(df, pred_col):
    correct = (df[pred_col] == df["label"])
    by_tag = correct.groupby(df["tag"]).mean().sort_values()
    by_tag["ALL"] = correct.mean()
    return by_tag.round(3)

In [4]:
acc_a = accuracy_by_tag(df, "pred_a")
assert "ALL" in acc_a.index and 0 <= acc_a["ALL"] <= 1
print("✅ TODO A1 passed")
acc_a

✅ TODO A1 passed


tag
sarcasm           0.500
slang             0.667
other-language    0.833
easy              1.000
mixed             1.000
negation          1.000
ALL               0.896
dtype: float64

### TODO A2 — the most *confident* mistakes
Confident mistakes are the dangerous ones: a user has no warning. Return the `n` wrong predictions with the highest confidence
(columns `text`, `label`, prediction, confidence).

In [5]:
def worst_errors(df, pred_col, conf_col, n=5):
    wrong = df[df[pred_col] != df["label"]]
    return wrong.sort_values(conf_col, ascending=False)[["text", "label", pred_col, conf_col]].head(n)

In [6]:
errs = worst_errors(df, "pred_a", "conf_a", n=5)
assert len(errs) <= 5 and (errs["label"] != errs["pred_a"]).all()
print("✅ TODO A2 passed")
print("Confusion matrix (rows = truth, cols = prediction; order NEGATIVE, POSITIVE):")
print(confusion_matrix(df.label, df.pred_a, labels=["NEGATIVE", "POSITIVE"]))
errs

✅ TODO A2 passed
Confusion matrix (rows = truth, cols = prediction; order NEGATIVE, POSITIVE):
[[24  2]
 [ 3 19]]


,text,label,pred_a,conf_a
27,"Wow, thirty minutes late again. Truly impressi...",NEGATIVE,POSITIVE,0.999764
26,"Oh great, another software update that breaks ...",NEGATIVE,POSITIVE,0.993503
32,"Lowkey obsessed with this new game, 10/10.",POSITIVE,NEGATIVE,0.984730
30,"This place slaps, the burgers are fire.",POSITIVE,NEGATIVE,0.947839
38,"Ein wunderbares Hotel, ich komme sehr gerne wi...",POSITIVE,NEGATIVE,0.892394


**Analysis A (write 4–6 sentences):** On which tags does model A fail and why? Is it *biased* toward one class? Do the most confident mistakes
share a pattern? What would you collect or change to fix them?

_Your analysis:_

## B · Evaluate model B and compare fairly
Model B predicts **1–5 stars**. To compare with the binary gold labels we must *decide* how to map stars to labels — and that decision is part of your result.

In [7]:
clf_b = pipeline("text-classification", model=MODEL_B, device=DEVICE)
res_b = clf_b(df.text.tolist(), batch_size=16, truncation=True)
df["raw_b"] = [r["label"] for r in res_b]            # e.g. "4 stars"
df["conf_b"] = [r["score"] for r in res_b]
df[["text", "label", "raw_b"]].head()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8057.10it/s]

,text,label,raw_b
0,"Absolutely loved this movie, the acting was br...",POSITIVE,5 stars
1,The hotel was spotless and the staff were incr...,POSITIVE,5 stars
2,"Best pizza I have had in years, we will defini...",POSITIVE,5 stars
3,Great battery life and the screen is gorgeous.,POSITIVE,5 stars
4,A heartwarming story that left the whole theat...,POSITIVE,5 stars


### TODO B1 — map stars to labels
4–5 stars → `POSITIVE`, 1–2 stars → `NEGATIVE`, 3 stars → `NEUTRAL` (the model *abstains*; a binary gold label can never match it).

In [8]:
def stars_to_label(raw):
    stars = int(raw.split()[0])
    return "POSITIVE" if stars >= 4 else "NEGATIVE" if stars <= 2 else "NEUTRAL"

In [9]:
assert [stars_to_label(s) for s in ["1 star", "3 stars", "5 stars"]] == ["NEGATIVE", "NEUTRAL", "POSITIVE"]
print("✅ TODO B1 passed")
df["pred_b"] = df.raw_b.map(stars_to_label)

acc_b = accuracy_by_tag(df, "pred_b")
coverage_b = (df.pred_b != "NEUTRAL").mean()
print(f"Model B abstained on {1 - coverage_b:.0%} of the reviews (counted as wrong above).")
compare = pd.DataFrame({"model A (DistilBERT, EN)": acc_a, "model B (BERT, multilingual)": acc_b})
compare

✅ TODO B1 passed
Model B abstained on 8% of the reviews (counted as wrong above).


,"model A (DistilBERT, EN)","model B (BERT, multilingual)"
tag,,
ALL,0.896,0.792
easy,1.000,1.000
mixed,1.000,0.667
negation,1.000,0.800
other-language,0.833,1.000
sarcasm,0.500,0.000
slang,0.667,0.667


**Analysis B (write 5–8 sentences):** Which model wins overall? On which tags does the ranking *flip*? Re-compute accuracy for model B
**only on the non-neutral predictions** — does the comparison change, and is that fair to model A? Which model would you ship for a French e-commerce site, and why?

_Your analysis:_

## C · Model-card critique
Fetch the facts programmatically, then read the two model cards on the Hub and complete the table in your report.

In [10]:
from huggingface_hub import HfApi
api = HfApi()
rows = []
for mid in [MODEL_A, MODEL_B]:
    info = api.model_info(mid)
    rows.append({"model": info.id, "task": info.pipeline_tag, "license": getattr(info.card_data, "license", None),
                 "downloads": info.downloads, "likes": info.likes, "last_modified": str(info.last_modified)[:10]})
pd.DataFrame(rows)

,model,task,license,downloads,likes,last_modified
0,distilbert/distilbert-base-uncased-finetuned-s...,text-classification,apache-2.0,2632606,986,2023-12-19
1,nlptown/bert-base-multilingual-uncased-sentiment,text-classification,mit,689767,495,2025-01-02


For each model answer (in your report): *what data was it trained on? what languages? what are the stated limitations / biases?
can you use it commercially? what is missing from the card that you would want to know?*

---
✅ **Done?** Save the notebook **with outputs** and submit it together with your report (see `assignment/ASSIGNMENT.md`).